In [32]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

torch.manual_seed(1337)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


In [33]:
if not Path("input.txt").exists():
    !wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

text = open("input.txt", "r", encoding="utf-8").read()
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

print("text length:", len(text))
print("vocab_size:", vocab_size)
print(text[:500])

text length: 1115394
vocab_size: 65
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [34]:
class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

block_size = 64

dataset = NextTokenDataset(data, block_size)

loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True
)

xb, yb = next(iter(loader))

print("xb.shape:", xb.shape)
print("yb.shape:", yb.shape)

xb.shape: torch.Size([64, 64])
yb.shape: torch.Size([64, 64])


In [35]:
class SingleHeadSelfAttention(nn.Module):
    def __init__(self, emb_dim, block_size):
        super().__init__()
        self.key = nn.Linear(emb_dim, emb_dim, bias=False)
        self.query = nn.Linear(emb_dim, emb_dim, bias=False)
        self.value = nn.Linear(emb_dim, emb_dim, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) * (C ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        out = wei @ v
        return out

class TinyAttentionLM(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=64):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)
        self.attn = SingleHeadSelfAttention(emb_dim, block_size)
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        tok = self.token_embedding(x)
        pos = self.position_embedding(pos)[None]
        h = tok + pos
        h = self.attn(h)
        logits = self.lm_head(h)
        return logits

model = TinyAttentionLM(vocab_size, block_size)
logits = model(xb)
print("logits.shape:", logits.shape)

logits.shape: torch.Size([64, 64, 65])


In [36]:
import torch
import torch.nn as nn
import torch.nn.functional as F

vocab_size = 65
block_size = 64
batch_size = 4

xb = torch.randint(0, vocab_size, (batch_size, block_size))

class Head(nn.Module):
    def __init__(self, emb_dim, head_size, block_size, dropout=0.1):
        super().__init__()
        self.key = nn.Linear(emb_dim, head_size, bias=False)
        self.query = nn.Linear(emb_dim, head_size, bias=False)
        self.value = nn.Linear(emb_dim, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2, -1)
        wei = wei * (k.size(-1) ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        head_size = emb_dim // num_heads
        self.heads = nn.ModuleList([
            Head(emb_dim, head_size, block_size, dropout) for _ in range(num_heads)
        ])
        self.proj = nn.Linear(emb_dim, emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

class FeedForward(nn.Module):
    def __init__(self, emb_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            nn.ReLU(),
            nn.Linear(4 * emb_dim, emb_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(emb_dim)
        self.sa = MultiHeadAttention(emb_dim, num_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(emb_dim)
        self.ffwd = FeedForward(emb_dim, dropout)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=128, num_heads=4, num_layers=4, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)
        self.blocks = nn.Sequential(*[
            Block(emb_dim, num_heads, block_size, dropout) for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)

        tok = self.token_embedding(x)
        pos = self.position_embedding(pos)[None]

        h = tok + pos
        h = self.blocks(h)
        h = self.ln_f(h)
        logits = self.lm_head(h)
        return logits

device = "cuda" if torch.cuda.is_available() else "cpu"

model = TinyGPT(vocab_size, block_size).to(device)
xb = xb.to(device)

logits = model(xb)

print("logits.shape:", logits.shape)

logits.shape: torch.Size([4, 64, 65])


In [37]:
import torch


def sequence_cross_entropy(logits, targets):
    return F.cross_entropy(logits.transpose(1, 2), targets)


def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()

    total_loss = 0.0
    total_count = 0

    for step, (xb, yb) in enumerate(loader):

        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)

        loss = sequence_cross_entropy(logits, yb)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)

        if max_steps is not None and step + 1 >= max_steps:
            break

    return total_loss / total_count


device = "cuda" if torch.cuda.is_available() else "cpu"

model = TinyGPT(vocab_size, block_size).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

print("model created")
print(device)

model created
cpu


In [43]:
for epoch in range(100):

    train_loss = train_one_epoch(
        model,
        loader,
        optimizer,
        device,
        max_steps=300
    )

    print(
        f"epoch {epoch:2d} | train loss {train_loss:.4f}"
    )

epoch  0 | train loss 2.6694


KeyboardInterrupt: 

In [38]:
@torch.no_grad()
def sample_gpt(
    model,
    block_size,
    stoi,
    itos,
    device,
    start_text="ROMEO:",
    max_new_tokens=400
):
    model.eval()

    context = torch.zeros(
        (1, block_size),
        dtype=torch.long,
        device=device
    )

    for ch in start_text:
        if ch in stoi:
            ix = torch.tensor(
                [[stoi[ch]]],
                device=device
            )
            context = torch.cat(
                [context[:, 1:], ix],
                dim=1
            )

    out = list(start_text)

    for _ in range(max_new_tokens):
        logits = model(context)
        logits = logits[:, -1, :]
        probs = F.softmax(
            logits,
            dim=-1
        )
        ix = torch.multinomial(
            probs,
            num_samples=1
        )
        out.append(
            itos[ix.item()]
        )
        context = torch.cat(
            [context[:, 1:], ix],
            dim=1
        )

    return "".join(out)


print(
    sample_gpt(
        model,
        block_size,
        stoi,
        itos,
        device,
        start_text="ROMEO:",
        max_new_tokens=500
    )
)

ROMEO:KgCI!KAQlCkAHkTd$.YUVgU EE.JKPFpsTpkJkp-fUAR,tRTc'zqLqKsX&p.UYI IVGP'wU,KYq:m!VjEUbf'ZLKmGk$MK, p3tDVVhO bBxDIKzx;yUpje!obWKdeipKtY:
zfLLFF:NdS'eE-pkqBm'I.v!JisOEYVW'kItTtkcIEmEOZOKhfErUb'aYbpAb..pY
z$YT:kQrizGKiGd!jE,!bEvDyFIhx
Ir;pIzVITP DL.tsqyIR-NV&:Dx,AIy,VHy!?p-$xA!VEiS&nXT;XZPDQII:XEk,Hzv'VAtZS&$'&eQVuQi,
BBpUqED&rYy:dcpiUKkXV-'qXidFTad3:upK.fpkH
pWcQdPgI'osvkcKkl&Op$YTsxt$xKKzMWXftheSHXXYWtESTSvWVxSpDn.zLNiw&UhLCEiANBW!KVAcpZmDtrnnHR wvfVvFi.dQogqipKcI,efkDA-IAlSiLFNYgi Vd!pEogsC:rxuTi?.


In [ ]:
hong_text = """
홍길동: 아버지를 아버지라 부르지 못하고 형을 형이라 부르지 못하니 어찌 사람답게 살 수 있겠습니까.
홍판서: 네 처지를 모르지 않으나 세상의 법도가 그러하니 경솔히 움직이지 말아라.
홍길동: 저는 이 집을 떠나 넓은 세상으로 나아가겠습니다.
춘섬: 도련님, 부디 몸조심하십시오.
홍길동: 힘없는 백성을 돕고 탐관오리를 벌하겠습니다.
도적: 우리를 이끌어 주시겠습니까.
홍길동: 의롭지 못한 재물은 거두고 백성에게 나누어 주겠다.
백성: 활빈당이 나타나 곡식을 나누어 주었다.
홍길동: 사람은 신분이 아니라 행동으로 평가받아야 한다.
왕: 홍길동이라는 자가 백성들 사이에서 이름을 떨친다.
""".strip()

In [39]:
chars_h = sorted(list(set(hong_text)))

stoi_h = {ch: i for i, ch in enumerate(chars_h)}

itos_h = {i: ch for ch, i in stoi_h.items()}

vocab_size_h = len(chars_h)

data_h = torch.tensor([stoi_h[ch] for ch in hong_text], dtype=torch.long)

dataset_h = NextTokenDataset(data_h, block_size)

loader_h = DataLoader(dataset_h, batch_size=64, shuffle=True)

print("hong vocab:", vocab_size_h)

hong vocab: 115


In [ ]:
model_h = TinyGPT(vocab_size_h, block_size).to(device)

optimizer_h = torch.optim.AdamW(model_h.parameters(), lr=3e-4)

for epoch in range(100):

    train_loss = train_one_epoch(
        model_h,
        loader_h,
        optimizer_h,
        device,
        max_steps=300
    )

    print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

In [42]:
print(
    sample_gpt(
        model_h,
        block_size,
        stoi_h,
        itos_h,
        device,
        start_text="홍길동:",
        max_new_tokens=300
    )
)

홍길동:: 아하,분이까성 행세빈관:  길받솔
홍동넓한러끌식 사니으몸성넓하 말신 아친말형 네 움네 돕를는 그왕않하저버 습있 두받평지심타집르돕주습움동
 이 주을고니으버심평 주
 리동라리
을  곡말디솔주고심 곡행님없그아나 오식친를왕길몸야 백판이떨 받한리나련시습나백들버직으으두상아이당 힘적
님 탐.
홍도빈조 르성직은지 름길왕하 
두시움탐받 못 러겠당 동으리홍가습못심말곡 않 말습자법겠 벌니버하라동길 길겠롭름세라 이 활도물로니 움.
도: 법식.롭식를지 시
게솔 오길지 물은롭빈누부습두물.행 이 빈 움주은지길:는 .
 평로으끌행었심 끌를직찌겠습라름행리하으
